

# The Graphical Method - Wyndor Glass
### OPIM 5641 - Business Decision Modeling · Module 2.2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5641-notebooks/blob/main/4_Graphical/g_GraphicalMethod_Wyndor.ipynb)

*Run me top to bottom - **Runtime → Run all**. Nothing to install, nothing to upload.*

______________________________________

We already solved Wyndor Glass with **brute force** - we made the computer try all 1,681 combinations and it came back with \$36,000. Now we're going to solve the *exact same problem* by drawing a picture, and we should land on the exact same answer. That's the point of using one problem over and over: **brute force, graphical, then Simplex** - three methods, one problem, so you can watch the ideas line up.

Take out a piece of paper and draw this while it's explained to you. Then try to replicate it in Python (looking at the code on your second monitor, and then without looking!). If you can do that, you really know it - otherwise you're just going through the motions.

---------------------------------------------------------------------------

**Wyndor Glass Co.** (Hillier & Lieberman, Chapter 3) makes two products in three plants:

* **Product 1:** an 8-foot glass door with aluminum framing (made in Plants 1 and 3)
* **Product 2:** a 4x6 foot double-hung wood frame window (made in Plants 2 and 3)

| | Product 1 ($x_1$) | Product 2 ($x_2$) | Hours available |
|---|---|---|---|
| Plant 1 | 1 | 0 | 4 |
| Plant 2 | 0 | 2 | 12 |
| Plant 3 | 3 | 2 | 18 |
| **Profit per batch** | \$3,000 | \$5,000 | |

### The linear program
Maximize $Z = 3000x_1 + 5000x_2$

subject to:
* $x_1 \le 4$ `(Plant 1)`
* $2x_2 \le 12$ `(Plant 2)`
* $3x_1 + 2x_2 \le 18$ `(Plant 3)`
* $x_1, x_2 \ge 0$ `(you can't make a negative number of doors)`

🔷 **The nugget:** the graphical method only works with **two** decision variables - but that limitation is exactly what makes it teachable. You can *see* the feasible region, and once you've seen it, Simplex stops being magic and starts being arithmetic that walks the same corners.

## Import Modules

In [ ]:
%matplotlib inline
from pylab import *   # for graphing - gives us plot, axis, fill_between and friends

# Plotting the Constraints on the Graph

To plot a constraint, take its equation and **set one variable to zero at a time** - that gives you two points, and two points make a line.

### Constraints
* $x_1 \le 4$ `(Plant 1)`
  * this one has no $x_2$ in it at all, so it's a **vertical line** at $x_1 = 4$
* $2x_2 \le 12$ `(Plant 2)`
  * divide both sides by 2: $x_2 \le 6$ - no $x_1$, so it's a **horizontal line** at $x_2 = 6$
* $3x_1 + 2x_2 \le 18$ `(Plant 3)`
  * when $x_1 = 0$: $2x_2 = 18$, so $x_2 = 9$
  * when $x_2 = 0$: $3x_1 = 18$, so $x_1 = 6$
  * so the line runs from $(0, 9)$ to $(6, 0)$

**Remember:** two of these three constraints are just straight vertical and horizontal lines. That happens whenever a product doesn't touch a plant at all - Plant 1 never makes windows, Plant 2 never makes doors.

We shade the side of each line that is **NOT allowed**, so the white space left in the middle is the region where every constraint is satisfied.

In [ ]:
figure(figsize=(7,7))
subplot(111, aspect='equal')
axis([0,10,0,10])
xlabel('Batches of Product 1 (doors), $x_1$')
ylabel('Batches of Product 2 (windows), $x_2$')

# Plant 1 constraint: x1 <= 4  (a VERTICAL line)
plot([4,4], [0,10], 'r', lw=2)
fill_between([4,10],          # x area to the right of the line
             [0,0],           # lower y
             [10,10],         # upper y
             color='red', alpha=0.15)

# Plant 2 constraint: 2*x2 <= 12  ->  x2 <= 6  (a HORIZONTAL line)
plot([0,10], [6,6], 'blue', lw=2)
fill_between([0,10],
             [6,6],
             [10,10],
             color='blue', alpha=0.15)

# Plant 3 constraint: 3*x1 + 2*x2 <= 18  (the slanted one)
plot([0,6], [9,0], 'green', lw=2)
fill_between([0,6,10],
             [9,0,0],
             [10,10,10],
             color='green', alpha=0.15)

title('Wyndor Glass - the three plant constraints')
show()

# The Feasible Region

The white area in the bottom-left is the **feasible region** - every point in there satisfies all three plant constraints (and both non-negativity constraints).

Every one of those points is a production plan we could actually run. But they are not equally good! Let's plug one in and see. How about $(x_1 = 1, x_2 = 2)$ - one batch of doors, two batches of windows?

$Z = 3000x_1 + 5000x_2$

$Z = 3000(1) + 5000(2)$

$Z = 13{,}000$

Let's put that point on the graph so we can see where it sits.

In [ ]:
figure(figsize=(7,7))
subplot(111, aspect='equal')
axis([0,10,0,10])
xlabel('Batches of Product 1 (doors), $x_1$')
ylabel('Batches of Product 2 (windows), $x_2$')

# Plant 1 constraint: x1 <= 4  (a VERTICAL line)
plot([4,4], [0,10], 'r', lw=2)
fill_between([4,10],          # x area to the right of the line
             [0,0],           # lower y
             [10,10],         # upper y
             color='red', alpha=0.15)

# Plant 2 constraint: 2*x2 <= 12  ->  x2 <= 6  (a HORIZONTAL line)
plot([0,10], [6,6], 'blue', lw=2)
fill_between([0,10],
             [6,6],
             [10,10],
             color='blue', alpha=0.15)

# Plant 3 constraint: 3*x1 + 2*x2 <= 18  (the slanted one)
plot([0,6], [9,0], 'green', lw=2)
fill_between([0,6,10],
             [9,0,0],
             [10,10,10],
             color='green', alpha=0.15)

# our trial plan: 1 batch of doors, 2 batches of windows
plot(1, 2, 'ko', markersize=9)
annotate('(1, 2) -> $13,000', xy=(1,2), xytext=(1.3,1.4), fontsize=11)

title('A feasible plan - but is it the BEST one?')
show()

Try another one. How about $(x_1 = 3, x_2 = 4)$?

$Z = 3000(3) + 5000(4) = 9{,}000 + 20{,}000 = 29{,}000$

Better! But we could keep guessing points forever - and that's exactly the brute-force trap we just escaped. There has to be a smarter way to know *where to look*.

# Find the Corner Points and Plot Them

Here's the idea that saves us: **the corner point property.** An optimal solution to an LP will always occur at a **corner point** of the feasible region. Corner points are the extreme points - where constraints cross each other, or cross an axis.

So instead of checking infinitely many points, we only check a handful.

To find a corner, you solve for the **intersection of two constraint equations**. Let's do all of them, walking **clockwise from the origin, smallest $x_1$ first** - the same order I want on your handwritten work:

* **O = (0, 0)** - the origin, where both axes meet
* **A = (0, 6)** - the $x_2$ axis meets the Plant 2 line ($x_2 = 6$)
* **B = (2, 6)** - Plant 2 meets Plant 3. Substitute $x_2 = 6$ into $3x_1 + 2x_2 = 18$: $3x_1 + 12 = 18$, so $3x_1 = 6$, so $x_1 = 2$
* **C = (4, 3)** - Plant 1 meets Plant 3. Substitute $x_1 = 4$ into $3x_1 + 2x_2 = 18$: $12 + 2x_2 = 18$, so $2x_2 = 6$, so $x_2 = 3$
* **D = (4, 0)** - the Plant 1 line meets the $x_1$ axis

**On your own:** do the algebra for points B and C by hand on paper before you look at my numbers. That substitution IS the skill - it's what the weekly check asks for.

In [ ]:
figure(figsize=(7,7))
subplot(111, aspect='equal')
axis([0,10,0,10])
xlabel('Batches of Product 1 (doors), $x_1$')
ylabel('Batches of Product 2 (windows), $x_2$')

# Plant 1 constraint: x1 <= 4  (a VERTICAL line)
plot([4,4], [0,10], 'r', lw=2)
fill_between([4,10],          # x area to the right of the line
             [0,0],           # lower y
             [10,10],         # upper y
             color='red', alpha=0.15)

# Plant 2 constraint: 2*x2 <= 12  ->  x2 <= 6  (a HORIZONTAL line)
plot([0,10], [6,6], 'blue', lw=2)
fill_between([0,10],
             [6,6],
             [10,10],
             color='blue', alpha=0.15)

# Plant 3 constraint: 3*x1 + 2*x2 <= 18  (the slanted one)
plot([0,6], [9,0], 'green', lw=2)
fill_between([0,6,10],
             [9,0,0],
             [10,10,10],
             color='green', alpha=0.15)

# the five corner points of the feasible region, clockwise from the origin
corners = [(0,0), (0,6), (2,6), (4,3), (4,0)]
labels  = ['O', 'A', 'B', 'C', 'D']

for (px, py), lab in zip(corners, labels):
    plot(px, py, 'ko', markersize=9)
    annotate(lab + f' ({px}, {py})', xy=(px,py), xytext=(px+0.25, py+0.25), fontsize=12)

title('The five corner points of the feasible region')
show()

# Evaluation of Corner Points

Recall our objective function:

$Max(Z) = 3000x_1 + 5000x_2$ `objective function`

Now we plug each corner point in, one at a time, and see which one pays the most:

* **Point O: $(x_1 = 0, x_2 = 0)$**
  * $Z = 3000(0) + 5000(0) = \$0$ *(make nothing, earn nothing)*
* **Point A: $(x_1 = 0, x_2 = 6)$**
  * $Z = 3000(0) + 5000(6) = \$30{,}000$
* **Point B: $(x_1 = 2, x_2 = 6)$**
  * $Z = 3000(2) + 5000(6) = 6{,}000 + 30{,}000 = \$36{,}000$ **[winner!]**
* **Point C: $(x_1 = 4, x_2 = 3)$**
  * $Z = 3000(4) + 5000(3) = 12{,}000 + 15{,}000 = \$27{,}000$
* **Point D: $(x_1 = 4, x_2 = 0)$**
  * $Z = 3000(4) + 5000(0) = \$12{,}000$

So the optimal plan is **2 batches of doors and 6 batches of windows, for \$36,000 per week.**

In [ ]:
# let's have Python check our arithmetic - and mark the winner on the graph
corners = [(0,0), (0,6), (2,6), (4,3), (4,0)]
labels  = ['O', 'A', 'B', 'C', 'D']

best_Z = 0
best_point = None
for (x1, x2), lab in zip(corners, labels):
    Z = 3000*x1 + 5000*x2                  # the objective function
    print(f'{lab}: ({x1}, {x2}) -> Z = ${Z:,}')
    if Z > best_Z:                         # is this corner the best so far?
        best_Z = Z
        best_point = (x1, x2, lab)

print()
print(f'WINNER: point {best_point[2]} at ({best_point[0]}, {best_point[1]}) with Z = ${best_Z:,}')

In [ ]:
figure(figsize=(7,7))
subplot(111, aspect='equal')
axis([0,10,0,10])
xlabel('Batches of Product 1 (doors), $x_1$')
ylabel('Batches of Product 2 (windows), $x_2$')

# Plant 1 constraint: x1 <= 4  (a VERTICAL line)
plot([4,4], [0,10], 'r', lw=2)
fill_between([4,10],          # x area to the right of the line
             [0,0],           # lower y
             [10,10],         # upper y
             color='red', alpha=0.15)

# Plant 2 constraint: 2*x2 <= 12  ->  x2 <= 6  (a HORIZONTAL line)
plot([0,10], [6,6], 'blue', lw=2)
fill_between([0,10],
             [6,6],
             [10,10],
             color='blue', alpha=0.15)

# Plant 3 constraint: 3*x1 + 2*x2 <= 18  (the slanted one)
plot([0,6], [9,0], 'green', lw=2)
fill_between([0,6,10],
             [9,0,0],
             [10,10,10],
             color='green', alpha=0.15)

# mark every corner, and put a star on the optimum
for (px, py), lab in zip(corners, labels):
    plot(px, py, 'ko', markersize=8)
    annotate(lab, xy=(px,py), xytext=(px+0.25, py+0.25), fontsize=12)

plot(2, 6, 'y*', markersize=26, markeredgecolor='black')
annotate('OPTIMAL: (2, 6) -> $36,000',
         xy=(2,6), xytext=(2.8,3.6),          # label sits in the clear feasible area
         fontsize=12, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='black', alpha=0.9),
         arrowprops=dict(arrowstyle='->', lw=1.6))

title('Wyndor Glass - optimal solution at corner B')
show()

# The Rosetta Stone check

Now flip back to the brute-force notebook. We looped over **1,681 combinations** and the computer told us: **\$36,000, making 2 of product 1 and 6 of product 2.**

Here, we drew three lines, found five corners, and did five multiplications on paper. **Same answer.**

That is worth sitting with for a second:

- **Brute force** checked 1,681 plans and had no idea what it was doing.
- **The graphical method** checked **5** plans - because the corner point property told us the optimum could only ever be at a corner.

We went from 1,681 to 5 by *understanding the geometry* instead of grinding. And when we get to **Simplex**, you'll see it does exactly what we just did by hand - hop from corner to corner, always uphill - except it works in as many dimensions as you like, long after we can't draw the picture anymore.

**Caution:** the graphical method is a two-variable party. Add a third product and you'd need a 3-D plot; add a fourth and there's nothing to draw at all. That's not a reason to skip it - it's the reason it's worth learning *first*, because the intuition carries into every method that follows.

**On your own:**
1. Plant 3 has 18 hours. Bump it to 24 and redraw. Which corner becomes optimal, and how much does the profit improve? *(That's sensitivity analysis, sneaking up on you early.)*
2. What happens to the picture if the profit on product 1 jumps from \$3,000 to \$8,000? Does the optimal corner move?
3. Solve **Veerman Furniture** graphically... or try to. What goes wrong, and what does that tell you about when to reach for this method?

## Bottom line

- The graphical method turns an LP into a **picture**: plot each constraint, shade what's not allowed, and the white space left over is the **feasible region**.
- The **corner point property** is the whole trick - the optimum always sits at a corner, so you only ever check a handful of points.
- Find corners by solving the **intersection of two constraints** (that's the algebra the weekly check wants), then plug each one into the objective and take the best.
- Wyndor graphically = **\$36,000 at (2, 6)** - the same answer brute force ground out over 1,681 combinations, from 5 points and a piece of paper.
- Next: the messy cases (redundant, infeasible, alternate optima, unbounded), and then **Simplex**, which walks these same corners in any number of dimensions.